## Example 1: Format dates and currencies by locale

In [ ]:
from datetime import date

from babel.dates import format_date
from babel.numbers import format_currency

# Example 1: Format dates and currencies by locale
day = date(2024, 3, 5)
us_date = format_date(day, locale="en_US")
de_date = format_date(day, locale="de_DE")
print(f"US date format: {us_date}")
print(f"German date format: {de_date}")
assert us_date == "Mar 5, 2024", us_date
assert de_date == "05.03.2024", de_date

amount = 1234.56
usd = format_currency(amount, "USD", locale="en_US")
eur = format_currency(amount, "EUR", locale="de_DE")
print(f"USD format: {usd}")
print(f"EUR format: {eur}")
assert usd == "$1,234.56", usd
assert eur == "1.234,56\xa0€", eur  # German puts the symbol after the amount, behind a no-break space

## Example 2: Locale display names and datetime formatting

In [ ]:
from datetime import datetime

from babel import Locale
from babel.dates import format_datetime, format_time

# Example 2: Locale display names and datetime formatting
locales = ["en_US", "fr_FR", "es_ES", "ja_JP", "ar_SA"]
for code in locales:
    locale = Locale.parse(code)
    print(f"  {code}: {locale.display_name} ({locale.get_display_name('en')})")
english_names = {code: Locale.parse(code).get_display_name("en") for code in locales}
assert english_names["fr_FR"] == "French (France)", english_names
assert english_names["ja_JP"] == "Japanese (Japan)", english_names
# display_name is the locale's name in its own language
assert Locale.parse("fr_FR").display_name == "français (France)"

moment = datetime(2024, 3, 5, 14, 30)
for code in locales[:3]:
    print(f"  {code}: {format_datetime(moment, locale=code)}")
assert format_datetime(moment, locale="fr_FR").startswith("5 mars 2024")

en_time = format_time(moment, format="short", locale="en_US")
de_time = format_time(moment, format="short", locale="de_DE")
print(f"  short time en_US: {en_time}  de_DE: {de_time}")
assert de_time == "14:30", de_time
assert en_time.startswith("2:30") and en_time.endswith("PM"), en_time  # 12-hour clock

## Example 3: Number, percent and unit formatting

In [ ]:
from babel.numbers import format_decimal, format_percent
from babel.units import format_unit

# Example 3: Number, percent and unit formatting
number = 1234567.89
formatted = {code: format_decimal(number, locale=code) for code in ["en_US", "de_DE", "fr_FR", "hi_IN"]}
for code, text in formatted.items():
    print(f"  {code}: {text}")
assert formatted["en_US"] == "1,234,567.89", formatted
assert formatted["de_DE"] == "1.234.567,89", formatted
assert formatted["hi_IN"] == "12,34,567.89", formatted  # Indian grouping: pairs of digits above the thousands
assert formatted["fr_FR"].endswith("567,89"), formatted  # French groups digits with a narrow no-break space

assert format_decimal(123.456, locale="de_DE") == "123,456"
assert format_percent(0.75, locale="en_US") == "75%"
assert format_percent(0.75, locale="es_ES") == "75\xa0%"
print("percent:", format_percent(0.75, locale="en_US"), "|", format_percent(0.75, locale="es_ES"))

distance = format_unit(25, "kilometer", locale="en_US")
print(f"Distance: {distance}")
assert distance == "25 kilometers", distance
assert format_unit(1, "length-kilometer", locale="en_US") == "1 kilometer"  # singular form

## Example 4: Message catalogs and locale data

In [ ]:
import babel.localedata as localedata
from babel.messages import Catalog

# Example 4: Message catalogs and locale data
catalog = Catalog(locale="en_US", domain="messages")
catalog.add("Hello World", string="Hello World")
catalog.add("Welcome", string="Welcome to our application")
catalog.add("Goodbye", string="See you later!")
print(f"Catalog locale: {catalog.locale}, messages: {len(catalog)}")
assert len(catalog) == 3
assert str(catalog.locale) == "en_US"
assert catalog["Welcome"].string == "Welcome to our application"
# Iterating yields the header entry (empty id) first, then the messages in the order they were added
assert [message.id for message in catalog] == ["", "Hello World", "Welcome", "Goodbye"]

locale_data = localedata.load("en_US")
assert locale_data["territories"]["DE"] == "Germany"
assert locale_data["languages"]["fr"] == "French"
print(f"en_US data names {len(locale_data['languages'])} languages and {len(locale_data['territories'])} territories")

# A catalog takes its plural rule from its locale
plural_rules = {code: (Catalog(locale=code).num_plurals, Catalog(locale=code).plural_expr)
                for code in ["en_US", "fr_FR", "ru"]}
print("plural rules:", plural_rules)
assert plural_rules["en_US"] == (2, "(n != 1)")
assert plural_rules["fr_FR"] == (2, "(n > 1)")
assert plural_rules["ru"][0] == 3

## Example 5: Default locale and CLDR locale parsing

In [ ]:
import babel
from babel import Locale

# Example 5: Default locale and CLDR locale parsing
print(f"Babel version: {babel.__version__}")

# default_locale() derives a locale from the environment: the given category, then LANGUAGE,
# LC_ALL, LC_CTYPE and LANG. It returns None when none of them names a locale.
default = babel.default_locale("LC_TIME")
print(f"Default LC_TIME locale: {default}")
assert default is None or isinstance(default, str), default

# Locale.parse maps a POSIX-style code to its CLDR identifier (zh_CN -> zh_Hans_CN)
common = ["en_US", "en_GB", "fr_FR", "de_DE", "es_ES", "zh_CN", "ja_JP"]
parsed = [str(Locale.parse(code)) for code in common]
print(f"Parsed locales: {parsed}")
assert parsed == ["en_US", "en_GB", "fr_FR", "de_DE", "es_ES", "zh_Hans_CN", "ja_JP"], parsed
assert str(Locale.parse("zh-TW", sep="-")) == "zh_Hant_TW"

# CLDR plural categories differ by language
assert [Locale("en").plural_form(n) for n in (1, 2)] == ["one", "other"]
assert [Locale("ru").plural_form(n) for n in (1, 3, 5)] == ["one", "few", "many"]